# Colab quickstart

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/AI-assisted-glioma-research/blob/main/notebooks/00_colab_quickstart.ipynb)

Open this notebook in Google Colab from GitHub, choose a CPU runtime
(**Runtime → Change runtime type → Hardware accelerator: None**) and run
**Runtime → Run all**. It clones this public repository (no GitHub token is
needed), installs the pinned uv and the locked Python 3.12 environment, can
fetch the released LUMIERE Patient-048 inputs for notebook 03, runs the notebook
checker and the synthetic demo, and offers the generated results for download.
Run elsewhere, for example on a Mac, each cell prints what to use instead and
skips its work without raising an error.

Notebooks 01–03 can also be opened directly in Colab, where they run in Colab's
own interpreter. This notebook instead runs them headless in the locked
environment, which is the closer check of reproducibility.

## AI contribution and verification

**Question:** How can the public repository reproduce these examples on Colab?

**AI contribution:** Research assistance with the Colab bootstrap and reproducibility documentation; this notebook installs and executes ordinary code.

**Investigator choices:** The investigator requested uv and Colab compatibility; cloud use and permitted inputs remain investigator decisions.

**Verification:** An earlier variant that uploaded a source archive of the private development repository completed every cell in one hosted Colab CPU session on 10 October 2026. This clone variant has not yet been run in hosted Colab.

**Evidence status:** No assistant or trained imaging model is called by the examples. The optional LUMIERE inputs are released derived data used under the LUMIERE authors' non-commercial condition.

These roles describe development and provenance. Rebuilding the examples uses ordinary code without an LLM call. [Assistance records](../docs/ai-workflow.md) retain the review status; record completeness is separate from scientific quality.

## Clone the repository
The shallow clone goes to `/content/AI-assisted-glioma-research`; the printed
revision identifies exactly what was run.

In [ ]:
import json
import subprocess
from pathlib import Path

try:
    from google.colab import files
except ModuleNotFoundError:
    files = None
IN_COLAB = files is not None
NOT_COLAB = "Skipped: not a hosted Google Colab session (see the first code cell)."
REPOSITORY = "https://github.com/arvidl/AI-assisted-glioma-research"

if not IN_COLAB:
    print(
        "This notebook is for hosted Google Colab; nothing was cloned, installed or run.\n"
        "On a local checkout, open notebooks 01-03 with the repository's .venv kernel,\n"
        "or run `make notebooks` and `make demo` from the repository root."
    )
else:
    ROOT = Path("/content/AI-assisted-glioma-research")
    if not ROOT.is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPOSITORY, str(ROOT)], check=True)
    revision = subprocess.run(
        ["git", "rev-parse", "HEAD"], cwd=ROOT, capture_output=True, text=True, check=True
    ).stdout.strip()
    print("Source revision:", revision)

## Install uv and use the locked Python environment
The isolated environment uses Python 3.12, downloading it if needed.
Scientific code runs in a subprocess, separate from Colab's base interpreter.
Colab's PYTHONPATH and MPLBACKEND are not passed on, and each command's output
is printed here so that failures are visible in the cell.

In [ ]:
import os
import shutil
import sys

ENV = {key: value for key, value in os.environ.items() if key not in {"PYTHONPATH", "MPLBACKEND"}}


def run(*args):
    result = subprocess.run(args, cwd=ROOT, env=ENV, capture_output=True, text=True)
    print(result.stdout, end="")
    print(result.stderr, end="", file=sys.stderr)
    result.check_returncode()


if not IN_COLAB:
    print(NOT_COLAB)
else:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "uv==0.12.23"],
        check=True,
    )
    UV = shutil.which("uv")
    if UV is None:
        raise RuntimeError("uv installation did not expose its executable.")
    run(UV, "sync", "--locked", "--group", "notebooks")
    run(UV, "run", "--locked", "glioma-ai", "doctor")

## Optional: fetch the LUMIERE inputs of notebook 03
With `FETCH_LUMIERE = True` the next cell downloads the release
`lumiere-p048-prepared-v1` (26 MB), checks its pinned SHA-256, extracts it to
`data/` and verifies every file; the checker then runs notebook 03. The files
are released derived data of LUMIERE Patient-048, used under the LUMIERE
authors' non-commercial condition and not covered by the MIT licence. Set
`FETCH_LUMIERE = False` to skip them; the checker then skips notebook 03.

In [ ]:
FETCH_LUMIERE = True

if not IN_COLAB:
    print(NOT_COLAB)
elif not FETCH_LUMIERE:
    print("LUMIERE inputs not fetched; the checker will skip notebook 03.")
else:
    run(UV, "run", "--locked", "glioma-ai", "lumiere-bundle-fetch", "--output", "data")
    ENV["LUMIERE_P048_ROOT"] = str(ROOT / "data/lumiere_p048_prepared")

## Run the notebooks and generate the synthetic figures
The notebook checker creates kernels using the locked interpreter and executes
notebooks 01–03 (03 only with the fetched inputs). The demo then writes the
synthetic example figures and records to `outputs/demo/`.

In [ ]:
if not IN_COLAB:
    print(NOT_COLAB)
else:
    run(UV, "run", "--locked", "--group", "notebooks", "python", "scripts/check_notebooks.py")
    print(json.loads((ROOT / "outputs/notebooks/execution.json").read_text()))
    run(UV, "run", "--locked", "glioma-ai", "demo", "--output", "outputs/demo")

In [ ]:
from IPython.display import Image, display

if not IN_COLAB:
    print(NOT_COLAB)
else:
    for name in ("imaging", "dynamics"):
        display(Image(filename=str(ROOT / "outputs/demo" / (name + ".png"))))

## Download results
This archive contains generated results and executed notebooks from `outputs/`.
Executed notebook 03 shows LUMIERE-derived figures under the authors'
non-commercial condition. Do not add clinical files to a cloud run without a
separate, appropriate data decision.

In [ ]:
if not IN_COLAB:
    print(NOT_COLAB)
else:
    archive_path = shutil.make_archive(
        str(ROOT.parent / "glioma-synthetic-results"), "zip", root_dir=ROOT / "outputs"
    )
    files.download(archive_path)